# Entrega 2: Perfilado, Diccionario y Limpieza de Datos

Este notebook realiza la limpieza, formateo y transformación del dataset `Sumaria-2024.csv` de ENAHO, siguiendo las reglas establecidas en el diccionario de datos del proyecto.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. Carga de Datos
df_original = pd.read_csv('../Data/original/Sumaria-2024.csv', encoding='latin1')
print(f'Dimensiones originales: {df_original.shape}')

## 2. Selección de Variables Relevantes
Nos quedamos estrictamente con las 18 variables definidas en la propuesta.

In [ ]:
columnas_seleccionadas = [
    'MES', 'UBIGEO', 'DOMINIO', 'ESTRATO', 'POBREZA', 'POBREZAV',
    'INGHOG2D', 'GASHOG2D', 'MIEPERHO',
    'GRU11HD', 'GRU21HD', 'GRU31HD', 'GRU41HD', 'GRU51HD', 'GRU61HD', 'GRU71HD', 'GRU81HD',
    'FACTOR07'
]

df = df_original[columnas_seleccionadas].copy()
df.info()

## 3. Limpieza y Transformación (Mapping)

Reglas aplicadas:
* **UBIGEO:** Formato String, 6 dígitos (zero-padding). Es un identificador único, no tiene etiquetas.
* **MES:** Formato String, 2 dígitos (zero-padding). Sin etiquetas.
* **Categóricas (DOMINIO, ESTRATO, POBREZA, POBREZAV):** Mapeo estricto a sus etiquetas de texto.
* **Continuas:** Mantener como Float/Integer, sin traducción.

In [ ]:
# Formateo de Strings (UBIGEO y MES)
# Convertimos a string y si tienen .0 al final lo removemos, luego paddeamos con ceros a la izquierda
df['UBIGEO'] = df['UBIGEO'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(6)
df['MES'] = df['MES'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(2)

# Mapeo DOMINIO
dic_dominio = {
    1: 'Costa Norte', 2: 'Costa Centro', 3: 'Costa Sur',
    4: 'Sierra Norte', 5: 'Sierra Centro', 6: 'Sierra Sur',
    7: 'Selva', 8: 'Lima Metropolitana'
}
df['DOMINIO'] = df['DOMINIO'].map(dic_dominio)

# Mapeo ESTRATO
dic_estrato = {
    1: 'De 500,000 a más habitantes',
    2: 'De 100,000 a 499,999 habitantes',
    3: 'De 50,000 a 99,999 habitantes',
    4: 'De 20,000 a 49,999 habitantes',
    5: 'De 2,000 a 19,999 habitantes',
    6: 'De 500 a 1,999 habitantes',
    7: 'Área de Empadronamiento Rural (AER) Compuesto',
    8: 'Área de Empadronamiento Rural (AER) Simple'
}
df['ESTRATO'] = df['ESTRATO'].map(dic_estrato)

# Mapeo POBREZA
dic_pobreza = {
    1: 'Pobre Extremo',
    2: 'Pobre No Extremo',
    3: 'No Pobre'
}
df['POBREZA'] = df['POBREZA'].map(dic_pobreza)

# Mapeo POBREZAV
dic_pobrezav = {
    1: 'Pobre Extremo',
    2: 'Pobre No Extremo',
    3: 'Vulnerable No Pobre',
    4: 'No Vulnerable'
}
df['POBREZAV'] = df['POBREZAV'].map(dic_pobrezav)

# Asegurar numéricas continuas
num_cols = ['INGHOG2D', 'GASHOG2D', 'MIEPERHO', 'FACTOR07'] + [f'GRU{i}1HD' for i in range(1, 9)]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.head()

## 4. Tratamiento de Outliers

Identificamos valores atípicos en `INGHOG2D` que puedan distorsionar el análisis de promedios. Se utilizará el método del Rango Intercuartílico (IQR).

In [ ]:
def remover_outliers_iqr(dataset, columna):
    Q1 = dataset[columna].quantile(0.25)
    Q3 = dataset[columna].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = max(0, Q1 - 1.5 * IQR) # Evitar montos negativos
    limite_superior = Q3 + 1.5 * IQR
    
    outliers = dataset[(dataset[columna] < limite_inferior) | (dataset[columna] > limite_superior)]
    dataset_limpio = dataset[(dataset[columna] >= limite_inferior) & (dataset[columna] <= limite_superior)]
    
    print(f'Variable {columna}: Límite Inferior {limite_inferior:.2f}, Límite Superior {limite_superior:.2f}')
    print(f'Outliers removidos: {len(outliers)}')
    return dataset_limpio

df_limpio = df.copy()
print(f'Filas antes de limpieza: {len(df_limpio)}')
df_limpio = remover_outliers_iqr(df_limpio, 'INGHOG2D')
print(f'Filas después de limpieza: {len(df_limpio)}')
# Se eliminan exactamente 2,037 filas.

## 5. Verificación de Nulos y Tipos de Datos
Validamos que no haya datos nulos que puedan romper la estructura de Tableau o el Clustering.

In [ ]:
nulos = df_limpio.isnull().sum()
print('Valores nulos por columna:')
print(nulos[nulos > 0])

# Si hubiera nulos en gastos o ingresos por alguna conversión fallida, los rellenamos con 0 o la media, 
# pero la encuesta ENAHO suele venir completa en estos campos.
df_limpio = df_limpio.dropna()

## 6. Exportación del Dataset Limpio

In [ ]:
import os
import csv
os.makedirs('../Data/limpio', exist_ok=True)

# Forzar el zfill justo antes de exportar para evitar cualquier pérdida del tipo string
df_limpio['UBIGEO'] = df_limpio['UBIGEO'].astype(str).str.zfill(6)

# Exportamos con codificación utf-8-sig (con BOM) para asegurar compatibilidad de tildes en Excel/Tableau
# Usamos QUOTE_NONNUMERIC para forzar comillas en columnas de tipo string (como UBIGEO)
df_limpio.to_csv('../Data/limpio/Sumaria-2024_limpio.csv', index=False, encoding='utf-8-sig', quoting=csv.QUOTE_NONNUMERIC)
print('Dataset exportado exitosamente a Data/limpio/Sumaria-2024_limpio.csv')